# 🌽 Build Android APK dengan Buildozer (Google Colab)

Notebook ini dibuat otomatis oleh Antigravity untuk membantu Anda membuat aplikasi **Corn Leaf Disease Classifier** (.apk) di cloud secara gratis tanpa setup WSL lokal di Windows Anda.

### 📋 Langkah-langkah Persiapan:
1. **Kompresi Folder Proyek**: Buat file `.zip` dari folder proyek `corn_disease_app` Anda di Windows. Beri nama **`corn_disease_app.zip`**.
   * *Catatan*: Pastikan file zip berisi berkas `main.py`, `corn_disease.kv`, `buildozer.spec`, folder `assets/`, dan folder `models/` (berisi bobot `.pkl` kustom).
2. **Jalankan Notebook**: Jalankan sel-sel di bawah ini secara berurutan.

## 🛠️ Langkah 1: Instalasi Dependensi Sistem Linux

In [ ]:
%%bash
# Tunggu hingga lock-frontend apt dirilis jika ada proses background lain
echo "Memeriksa kunci apt..."
while sudo fuser /var/lib/dpkg/lock-frontend >/dev/null 2>&1; do
   echo "Menunggu proses instalasi latar belakang selesai..."
   sleep 3
done

# Update dan install paket Linux yang dibutuhkan oleh toolchain Android
sudo apt-get update
sudo apt-get install -y git zip unzip colordiff libltdl-dev libffi-dev libssl-dev autoconf automake autotools-dev bison build-essential ccache libtool libtool-bin pkg-config python3-dev python3-pip python3-setuptools libsqlite3-dev sqlite3 libgdbm-dev libgdbm-compat-dev libdb-dev libc6-dev zlib1g-dev openjdk-17-jdk gettext libopenblas-dev libncurses5-dev libncursesw5-dev libtinfo5 cmake python3.10 python3.10-venv python3.10-dev
# Install JDK 17 yang dibutuhkan oleh Android SDK
sudo apt-get install -y openjdk-17-jdk

## 📦 Langkah 2: Instalasi Buildozer, Cython, dan Kloning Python-for-Android Custom

In [ ]:
# Install Buildozer dan Cython
!pip install --upgrade buildozer cython
# Periksa apakah buildozer terinstal dengan baik
!buildozer --version

# Verifikasi keberadaan perkakas libtool di sistem Colab
import shutil
import os
print("=== VERIFIKASI SISTEM ===")
print("Path libtoolize:", shutil.which("libtoolize"))
print("Path libtool:", shutil.which("libtool"))
print("Keberadaan /usr/share/aclocal/libtool.m4:", os.path.exists("/usr/share/aclocal/libtool.m4"))
print("=========================")

# Kloning repositori python-for-android ke direktori lokal Colab agar kita bisa melakukan patch pada resep libffi
if os.path.exists('/content/python-for-android'):
    shutil.rmtree('/content/python-for-android')
!git clone --depth 1 -b master https://github.com/kivy/python-for-android.git /content/python-for-android

# Patch resep libffi secara dinamis mendeteksi indentasi baris kode asli
recipe_path = '/content/python-for-android/pythonforandroid/recipes/libffi/__init__.py'
if os.path.exists(recipe_path):
    print("\nMemulai penambalan resep libffi secara dinamis...")
    with open(recipe_path, 'r') as f:
        lines = f.readlines()
    
    new_lines = []
    patched = False
    for line in lines:
        if "shprint(sh.Command('./autogen.sh')" in line:
            # Deteksi indentasi awal (spasi) dari baris shprint
            indent = line[:len(line) - len(line.lstrip())]
            print(f"Mendeteksi indentasi: {len(indent)} spasi.")
            
            # Susun kode patch dengan indentasi yang persis sama
            patch_code = [
                f"{indent}# Copy system m4 macros to local m4 directory\n",
                f"{indent}import shutil, glob, os\n",
                f"{indent}m4_dir = os.path.join(self.get_build_dir(arch.arch), 'm4')\n",
                f"{indent}os.makedirs(m4_dir, exist_ok=True)\n",
                f"{indent}print(f'Penyalinan macro sistem ke {{m4_dir}}...')\n",
                f"{indent}copied_files = []\n",
                f"{indent}for path in ['/usr/share/aclocal/*.m4', '/usr/local/share/aclocal/*.m4']:\n",
                f"{indent}    for m4_file in glob.glob(path):\n",
                f"{indent}        try:\n",
                f"{indent}            shutil.copy(m4_file, m4_dir)\n",
                f"{indent}            copied_files.append(os.path.basename(m4_file))\n",
                f"{indent}        except Exception:\n",
                f"{indent}            pass\n",
                f"{indent}print(f'=== PENYALINAN BERHASIL: {{len(copied_files)}} FILE MACRO DISALIN ===')\n",
                f"{indent}print(f'File macro yang disalin: {{copied_files}}')\n",
                f"{indent}if len(copied_files) == 0:\n",
                f"{indent}    print('[PERINGATAN SELESAI] 0 file macro disalin. Pastikan Langkah 1 dijalankan!')\n",
                f"{indent}env['ACLOCAL_PATH'] = '/usr/share/aclocal'\n",
            ]
            new_lines.extend(patch_code)
            new_lines.append(line)  # Masukkan kembali perintah shprint asli setelahnya
            patched = True
            print("Kode penambalan berhasil disisipkan dengan indentasi yang sesuai!")
        else:
            new_lines.append(line)
            
    if patched:
        with open(recipe_path, 'w') as f:
            f.writelines(new_lines)
        print("[SUKSES] Resep libffi python-for-android berhasil ditambal tanpa error indentasi!")
    else:
        print("[ERROR] Gagal menemukan target baris autogen.sh di resep libffi!")
else:
    print("[ERROR] Berkas resep libffi tidak ditemukan!")

## 📤 Langkah 3: Unggah dan Ekstrak File Proyek Anda
Jalankan sel di bawah ini, klik tombol **Choose Files**, lalu pilih file **`corn_disease_app.zip`** dari komputer Anda.

In [ ]:
from google.colab import files
import os
import shutil
import glob

# Hapus file zip lama di folder root Colab agar tidak ada penamaan ganda (e.g. corn_disease_app (1).zip)
print("Membersihkan sisa berkas lama...")
for f in glob.glob('corn_disease_app*.zip'):
    try:
        os.remove(f)
        print(f"Menghapus berkas zip lama: {f}")
    except Exception:
        pass

if os.path.exists('corn_disease_app'):
    try:
        shutil.rmtree('corn_disease_app')
        print("Folder corn_disease_app lama dihapus.")
    except Exception:
        pass

print("\nSilakan unggah berkas corn_disease_app.zip baru Anda:")
uploaded = files.upload()
zip_name = 'corn_disease_app.zip'

if zip_name in uploaded:
    !unzip -o {zip_name} -d corn_disease_app
    print("\n[SUKSES] Proyek berhasil diekstrak!")
else:
    # Antisipasi jika Colab tetap merename file yang diupload
    uploaded_keys = list(uploaded.keys())
    if uploaded_keys:
        actual_name = uploaded_keys[0]
        print(f"\nMendeteksi berkas terunggah dengan nama lain: {actual_name}")
        os.rename(actual_name, zip_name)
        !unzip -o {zip_name} -d corn_disease_app
        print("\n[SUKSES] Proyek berhasil diekstrak setelah rename!")
    else:
        print("\n[ERROR] Gagal mengunggah berkas. Silakan jalankan ulang sel ini!")

## 🔄 Langkah 3.5: Regenerasi Model TFLite Versi Lama (Kompatibel)
Sel ini akan menulis skrip konversi secara lokal di Colab, lalu membuat virtual environment Python 3.10 untuk menginstal TensorFlow 2.10.0 dan NumPy 1.x (yang kompatibel dengan Android Kivy) untuk mengonversi bobot `.pkl` menjadi model `.tflite` versi lama.

In [ ]:
import os

# Buat folder models jika belum ada
os.makedirs('/content/corn_disease_app/models', exist_ok=True)

# Tulis file convert_models_legacy.py langsung dari Colab untuk menjamin file selalu ada
with open('/content/corn_disease_app/convert_models_legacy.py', 'w') as f:
    f.write('''import tensorflow as tf
import pickle
import os

print("TensorFlow version inside venv310:", tf.__version__)

def build_cnn(input_shape=(224, 224, 3), num_classes=4):
    from tensorflow.keras import layers, models
    model = models.Sequential([
        layers.Rescaling(1./255, input_shape=input_shape),
        
        layers.Conv2D(32, (3, 3), activation=\'relu\'),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(64, (3, 3), activation=\'relu\'),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(128, (3, 3), activation=\'relu\'),
        layers.MaxPooling2D((2, 2)),
        
        layers.Conv2D(128, (3, 3), activation=\'relu\'),
        layers.MaxPooling2D((2, 2)),
        
        layers.Flatten(),
        layers.Dense(128, activation=\'relu\'),
        layers.Dropout(0.5),
        layers.Dense(num_classes, activation=\'softmax\')
    ])
    return model

def build_mobilenetv2(input_shape=(224, 224, 3), num_classes=4):
    from tensorflow.keras import layers, models
    base_model = tf.keras.applications.MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights=None
    )
    base_model.trainable = False
    
    model = models.Sequential([
        layers.Rescaling(scale=2./255, offset=-1., input_shape=input_shape),
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation=\'relu\'),
        layers.Dropout(0.3),
        layers.Dense(num_classes, activation=\'softmax\')
    ])
    return model

os.chdir(\'/content/corn_disease_app\')

# 1. Convert CNN Model
cnn_weights_path = "models/cnn_weights.pkl"
if os.path.exists(cnn_weights_path):
    print("Reconstructing CNN and loading weights from PKL...")
    cnn_model = build_cnn()
    with open(cnn_weights_path, "rb") as f:
        cnn_weights = pickle.load(f)
    cnn_model.set_weights(cnn_weights)
    
    print("Converting CNN model to legacy TFLite...")
    converter = tf.lite.TFLiteConverter.from_keras_model(cnn_model)
    tflite_model = converter.convert()
    with open("models/cnn_model.tflite", "wb") as f:
        f.write(tflite_model)
    print("Saved models/cnn_model.tflite successfully!")
else:
    print("CNN weights PKL not found at:", os.path.abspath(cnn_weights_path))

# 2. Convert MobileNetV2 Model
mobilenet_weights_path = "models/mobilenet_weights.pkl"
if os.path.exists(mobilenet_weights_path):
    print("Reconstructing MobileNetV2 and loading weights from PKL...")
    mobilenet_model = build_mobilenetv2()
    with open(mobilenet_weights_path, "rb") as f:
        mobilenet_weights = pickle.load(f)
    mobilenet_model.set_weights(mobilenet_weights)
    
    print("Converting MobileNetV2 model to legacy TFLite...")
    converter = tf.lite.TFLiteConverter.from_keras_model(mobilenet_model)
    tflite_model = converter.convert()
    with open("models/mobilenet_model.tflite", "wb") as f:
        f.write(tflite_model)
    print("Saved models/mobilenet_model.tflite successfully!")
else:
    print("MobileNetV2 weights PKL not found at:", os.path.abspath(mobilenet_weights_path))
''')

print("Script convert_models_legacy.py berhasil ditulis di /content/corn_disease_app/")

# Buat virtual environment Python 3.10
print("\nMembuat virtual environment Python 3.10...")
!python3.10 -m venv /content/venv310

# Install tensorflow==2.10.0 dan numpy<2 secara aman di dalam virtual environment 3.10
print("\nMemasang tensorflow==2.10.0 dan numpy<2 di venv310 (Proses ini memakan waktu ~1-2 menit)...")
!/content/venv310/bin/pip install -q --upgrade pip
!/content/venv310/bin/pip install -q tensorflow==2.10.0 "numpy<2"

# Jalankan skrip konversi di venv310
print("\nMenjalankan konversi model ke TFLite versi lama...")
!/content/venv310/bin/python /content/corn_disease_app/convert_models_legacy.py

## 🚀 Langkah 4: Memulai Proses Build APK
Proses ini akan memakan waktu sekitar **10-15 menit** pada percobaan pertama karena Buildozer akan mengunduh Android SDK, NDK, dan mengompilasi semua dependensi Python (Kivy, KivyMD, Pillow, Numpy, TFLite).

In [ ]:
%cd /content/corn_disease_app
# Mengatur environment variable ACLOCAL_PATH secara global di notebook agar autoconf/aclocal menemukan macro libtool
%env ACLOCAL_PATH=/usr/share/aclocal

# Jika build sebelumnya gagal, hapus tanda komentar '#' pada baris di bawah untuk membersihkan cache build:
# !buildozer android clean

# Menggunakan perintah 'yes' untuk menjawab 'y' (yes) pada semua prompt interaktif secara otomatis
!yes | buildozer -v android debug 2>&1 | tee build.log

### 🔍 Sel Diagnostik (Jalankan ini HANYA jika Langkah 4 di atas ERROR/GAGAL)
Jalankan sel di bawah ini untuk menampilkan 100 baris terakhir dari log build secara bersih tanpa terpotong.

In [ ]:
# Tampilkan 100 baris terakhir log
!tail -n 100 /content/corn_disease_app/build.log

## 📥 Langkah 5: Unduh APK Hasil Build
Jika build berhasil, APK akan tersimpan di dalam folder `bin/`. Jalankan sel di bawah ini untuk mengunduhnya ke PC Anda.

In [ ]:
from google.colab import files
import os

%cd /content/corn_disease_app
apk_file = None
if os.path.exists('bin'):
    for f in os.listdir('bin'):
        if f.endswith('.apk'):
            apk_file = os.path.join('bin', f)
            break

if apk_file:
    print(f"Mengunduh berkas APK: {apk_file}")
    files.download(apk_file)
else:
    print("[ERROR] Berkas APK tidak ditemukan di folder bin/. Periksa log build di atas untuk melihat kegagalan.")